# LSTM Experiment
- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
- **Owner:** Daniel Alejandro Castillo Martin
---


Starter notebook: shared data loading and batch verification are provided; model implementation, training, and results remain to be completed. No results are fabricated.

Reuse the prepared Enron partitions and training vocabulary. Select configurations on validation data and reserve test evaluation for frozen final models.


## 1. Initial Configuration


### 1.1. Imports, Paths, and Device


In [1]:
# Import required libraries
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

# Locate the repository from either its root or the notebooks folder
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'src').is_dir() else current_dir.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Place this notebook in the repository notebooks folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import shared project modules
from src.config import BATCH_SIZE, TRAINING_SEEDS
from src.data import load_prepared_data
from src.reproducibility import set_seed
from src.text import EncodedTextDataset, make_data_loader
from src.lstm_model import LSTMClassifier
from src.evaluation import classification_metrics, evaluate_model
from src.training import fit_model
from src.config import (TUNING_SEED, WEIGHT_DECAY, MAX_GRAD_NORM, SELECTION_METRIC)

# Use CUDA when it is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the runtime and selected device
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

c:\Users\dancm\OneDrive\Documentos\humber-aigc-postgraduate\aigc-5500-advanced-deep-learning\midterm-project\AIGC5500-Midterm-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10
Interpreter: c:\Users\dancm\OneDrive\Documentos\humber-aigc-postgraduate\aigc-5500-advanced-deep-learning\midterm-project\AIGC5500-Midterm-Project\.venv\Scripts\python.exe
PyTorch: 2.9.1+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### 1.2. Experiment Configuration


In [2]:
# Name this experiment and its owner
MODEL_NAME = 'lstm'
OWNER = 'Daniel Alejandro Castillo Martin'

# Reset randomness before the first run
set_seed(TRAINING_SEEDS[0])

# Point to the shared prepared data and this model's output folders
PREPARED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'enron'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'enron' / MODEL_NAME
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'enron' / MODEL_NAME

# Show the shared training seeds and batch size
print('Training seeds:', TRAINING_SEEDS)
print('Batch size:', BATCH_SIZE)

Training seeds: (42, 123, 2026)
Batch size: 64


## 2. Load Shared Prepared Data


Run `01_data_preparation.ipynb` once before this notebook. The saved preprocessing metadata controls sequence length; this notebook does not refit the vocabulary.


In [3]:
# Load the saved splits, vocabulary, and metadata
prepared = load_prepared_data(PREPARED_DATA_DIR)

# Keep the cleaned partitions
splits = prepared['splits']

# Keep the training vocabulary
vocabulary = prepared['vocabulary']

# Keep the preprocessing configuration
metadata = prepared['metadata']

# Use the saved sequence length instead of choosing a new one
MAX_LENGTH = metadata['max_length']

# Confirm the vocabulary size and reserved token IDs
assert len(vocabulary) == metadata['vocabulary_size']
assert vocabulary['<pad>'] == 0 and vocabulary['<unk>'] == 1
# Confirm each split still has the saved number of rows
for name, expected_size in metadata['split_sizes'].items():
    assert len(splits[name]) == expected_size

# Show the loaded split sizes
pd.DataFrame([{'split': name, 'rows': len(part)} for name, part in splits.items()])

,split,rows
0,train,22524
1,validation,5632
2,test,1981


In [4]:
# Print dataset information
print('Vocabulary size:', len(vocabulary))
print('Maximum length:', MAX_LENGTH)
print('Dataset:', metadata['dataset_id'])
print('Dataset revision:', metadata['dataset_revision'])

Vocabulary size: 10000
Maximum length: 256
Dataset: SetFit/enron_spam
Dataset revision: main


### 2.1. Encode Development Partitions


Only training and validation are encoded. Recreate the training loader for each seed; keep validation order fixed.


In [5]:
# Encode train and validation only
encoded_datasets = {
    name: EncodedTextDataset(
        texts = splits[name]['text'],
        labels = splits[name]['label'],
        vocabulary = vocabulary,
        max_length = MAX_LENGTH) for name in ('train', 'validation')
    }

# Keep validation order fixed
validation_loader = make_data_loader(encoded_datasets['validation'], batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

# Inspect one validation batch
example_batch = next(iter(validation_loader))
for name, tensor in example_batch.items():
    print(name, tuple(tensor.shape))

# Recreate the training loader inside every seeded run, before training.
# Do not encode or evaluate the test partition during development.

tokens (64, 256)
lengths (64,)
labels (64,)


## 3. LSTM Architecture


- The classifier uses learned 64-dimensional token embeddings and a single unidirectional LSTM with 128 hidden units. Packed sequences exclude padding from recurrent processing.

- The final hidden state represents each email. Dropout is applied before a linear layer produces two raw scores: non-spam and spam.

- The main model initializes the LSTM biases to zero. The proposed ablation changes only the effective forget-gate bias from zero to one.

In [6]:
# Define the model configuration
MODEL_CONFIG = {'embed_dim': 64, 'hidden_dim': 128, 'dropout': 0.1}

# Set seed for reproducibility
set_seed(TRAINING_SEEDS[0])

# Define the model
model = LSTMClassifier(vocab_size = len(vocabulary), **MODEL_CONFIG, forget_bias = 0.0).to(device)

# Calculate the number of trainable parameters
parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

# Print the model and the number of trainable parameters
print(model)
print(f'Trainable parameters: {parameter_count:,}')

LSTMClassifier(
  (embedding): Embedding(10000, 64, padding_idx=0)
  (lstm): LSTM(64, 128, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)
Trainable parameters: 739,586


### 3.1. Verify Output Shape and Padding Handling

The untrained model is checked for finite outputs and the expected two-score output shape. Additional right padding must not change its predictions when the original sequence lengths are preserved.

In [7]:
# Evaluate the model
model.eval()

# Get the tokens and lengths from the example batch
tokens = example_batch['tokens'].to(device)
lengths = example_batch['lengths']

# Evaluate the model with inference mode
with torch.inference_mode():
    logits = model(tokens, lengths)
    
# Check the output shape and the finiteness of the logits
assert logits.shape == (len(lengths), 2)
assert torch.isfinite(logits).all().item()

# Print the output shape and the first three score pairs
print('Output shape:', logits.shape)
print('First three score pairs:')
print(logits[:3].cpu())

Output shape: torch.Size([64, 2])
First three score pairs:
tensor([[ 0.1027,  0.0393],
        [ 0.1257,  0.0387],
        [-0.0487,  0.0672]])


In [8]:
# Append ten padding positions without changing the real lengths.
extra_padding = torch.zeros((tokens.shape[0], 10), dtype = tokens.dtype, device = device)

# Concatenate the extra padding to the tokens
padded_tokens = torch.cat([tokens, extra_padding], dim = 1)

# Evaluate the model with the padded tokens
with torch.inference_mode():
    padded_logits = model(padded_tokens, lengths)

# Check if the logits are close to the original logits
torch.testing.assert_close(logits, padded_logits, rtol = 1e-4, atol = 1e-5)

# Print the result
print('Padding check passed: extra padding does not change the scores.')

Padding check passed: extra padding does not change the scores.


### 4.1. Verify Evaluation Metrics

A small example with known predictions verifies the shared metric calculation before it is used for model validation. Both macro F1 and spam F1 are reported explicitly to avoid confusing the two measures.

In [9]:
# Test the classification metrics
example_true = [0, 0, 1, 1]
example_pred = [0, 0, 0, 1]

example_metrics = classification_metrics(example_true, example_pred)

# Display the example metrics
display(pd.Series(example_metrics))

# Assert the example metrics are correct with a small tolerance for floating point imprecision
assert abs(example_metrics['accuracy'] - 0.75) < 1e-9
assert abs(example_metrics['macro_f1'] - 0.7333333333333334) < 1e-9
assert abs(example_metrics['spam_f1'] - 0.6666666666666666) < 1e-9
assert abs(example_metrics['spam_precision'] - 1.0) < 1e-9
assert abs(example_metrics['spam_recall'] - 0.5) < 1e-9

print('Metric checks passed.')

accuracy          0.750000
macro_f1          0.733333
spam_f1           0.666667
spam_precision    1.000000
spam_recall       0.500000
dtype: float64

Metric checks passed.


### 4.2. Development Training Run

A one-epoch development run checks that the LSTM, optimizer, validation procedure, and checkpoint saving work together. This is a pipeline check, not a completed tuning or final comparison experiment

In [10]:
# Test the classification metrics
example_true = [0, 0, 1, 1]
example_pred = [0, 0, 0, 1]

example_metrics = classification_metrics(
    example_true,
    example_pred,
)

# Display the example metrics
display(pd.Series(example_metrics))

# Assert the example metrics are correct with a small tolerance for floating point imprecision
assert abs(example_metrics['accuracy'] - 0.75) < 1e-9
assert abs(example_metrics['macro_f1'] - 0.7333333333333334) < 1e-9
assert abs(example_metrics['spam_f1'] - 0.6666666666666666) < 1e-9
assert abs(example_metrics['spam_precision'] - 1.0) < 1e-9
assert abs(example_metrics['spam_recall'] - 0.5) < 1e-9

print('Metric checks passed.')

accuracy          0.750000
macro_f1          0.733333
spam_f1           0.666667
spam_precision    1.000000
spam_recall       0.500000
dtype: float64

Metric checks passed.


In [11]:
# Development run
development_run = fit_model(
    model_factory = lambda: LSTMClassifier(
        vocab_size = len(vocabulary),
        **MODEL_CONFIG,
        forget_bias = 0.0,
    ),
    train_dataset=encoded_datasets["train"],
    validation_loader = validation_loader,
    device = device,
    seed = TUNING_SEED,
    epochs = 1,
    learning_rate = 1e-3,
    weight_decay = WEIGHT_DECAY,
    max_grad_norm = MAX_GRAD_NORM,
    batch_size = BATCH_SIZE,
    checkpoint_path = CHECKPOINT_DIR / "development_seed42_epoch1.pt",
    history_path = RESULTS_DIR / "development_seed42_epoch1.csv",
    selection_metric = SELECTION_METRIC,
)

# Display the history
display(development_run["history"])
# Print the training time
print(f"Training time: {development_run['training_seconds']:.1f} seconds")

Epoch 01/1 | train loss 0.3134 | val loss 0.2171 | val macro_f1 0.9264


,epoch,seed,learning_rate,train_loss,val_accuracy,val_macro_f1,val_spam_f1,val_spam_precision,val_spam_recall,val_loss
0,1,42,0.001,0.313441,0.926491,0.926449,0.928225,0.875695,0.987459,0.21715


Training time: 10.3 seconds
